# Sensor System Capture quick look
Set `analysis_dir` to the directory produced by `capture-export`. Raw axes remain in native ADXL355 signed sample codes.

In [ ]:
# ADXL355 configuration
RANGE_G = 2  # Zmień na 4 lub 8, jeśli taki zakres ustawiono

SENSITIVITY = {
    2: 256_000,
    4: 128_000,
    8: 64_000
}

G = 9.80665
LSB_TO_MS2 = G / SENSITIVITY[RANGE_G]

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

analysis_dir = Path('../../var/tmp/manual-rs485-hil-20261009T140007Z/analysis')
samples = np.genfromtxt(analysis_dir / 'samples.csv', delimiter=',', names=True)
events = np.atleast_1d(np.genfromtxt(analysis_dir / 'events.csv', delimiter=',', names=True, dtype=None, encoding='utf-8'))
metadata = json.loads((analysis_dir / 'metadata.json').read_text())

time_s = (samples['device_time_us_estimate'] - samples['device_time_us_estimate'][0]) / 1e6
dt = np.diff(samples['device_time_us_estimate']) / 1e6
effective_odr_hz = 1.0 / np.median(dt[dt > 0])
metadata, effective_odr_hz, events

In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(14, 8))
for axis, field, label in zip(axes, ['x_raw', 'y_raw', 'z_raw'], ['X raw', 'Y raw', 'Z raw']):
    axis.plot(time_s, samples[field], linewidth=0.7)
    axis.set_ylabel(label)
    axis.grid(True)
axes[-1].set_xlabel('Device time from first sample [s]')
fig.suptitle(f"{metadata['hardware_id']} — {effective_odr_hz:.4f} Hz estimated")
fig.tight_layout()

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuration
WINDOW_S = 8.0
STEP_S = 1.0

SEGMENT_START_S = 0.0
SEGMENT_DURATION_S = 580.0

FS = float(effective_odr_hz)
CHANNELS = ["x_raw", "y_raw", "z_raw"]

RANGE_G = 2
SENSITIVITY = {2: 256_000, 4: 128_000, 8: 64_000}
SCALE = 9.80665 / SENSITIVITY[RANGE_G]

# Window positions
max_start = max(0.0, float(time_s[-1]) - WINDOW_S)
segment_start = float(np.clip(SEGMENT_START_S, 0, max_start))
segment_end = min(max_start, segment_start + SEGMENT_DURATION_S)

starts = np.arange(
    segment_start,
    segment_end + STEP_S / 2,
    STEP_S
)
starts = starts[starts <= max_start + 1e-9]


def window_traces(start_s):
    mask = (time_s >= start_s) & (time_s < start_s + WINDOW_S)

    t = time_s[mask] - start_s
    raw = np.column_stack([
        samples[ch][mask] for ch in CHANNELS
    ])

    n = len(t)

    if n < 4:
        return [go.Scatter(x=[], y=[]) for _ in range(9)], \
               "Not enough samples"

    repeated = np.all(raw[1:] == raw[:-1], axis=1)
    gaps = np.sum(np.diff(t) > 1.5 / FS)

    freq = np.fft.rfftfreq(n, d=1 / FS)
    window = np.hanning(n)

    traces = []

    for i in range(3):
        y = raw[:, i].astype(float) * SCALE

        # Time domain
        traces.append(go.Scatter(
            x=t,
            y=y,
            mode="lines+markers",
            marker=dict(size=4),
            line=dict(width=1)
        ))

        # Repeated samples
        traces.append(go.Scatter(
            x=t[1:][repeated],
            y=y[1:][repeated],
            mode="markers",
            marker=dict(color="red", size=8)
        ))

        # FFT
        signal = y - np.mean(y)
        spectrum = np.fft.rfft(signal * window)

        amplitude = 2 * np.abs(spectrum) / np.sum(window)
        amplitude[0] = 0

        if n % 2 == 0:
            amplitude[-1] /= 2

        traces.append(go.Scatter(
            x=freq,
            y=amplitude,
            mode="lines"
        ))

    title = (
        f"Window: {start_s:.2f}–{start_s + WINDOW_S:.2f} s"
        f" | Samples: {n}"
        f" | Repeated XYZ: {int(np.sum(repeated))}"
        f" | Gaps: {int(gaps)}"
    )

    return traces, title


# Initial figure
initial_traces, initial_title = window_traces(starts[0])

fig = make_subplots(
    rows=3,
    cols=2,
    shared_xaxes=True,
    vertical_spacing=0.08,
    horizontal_spacing=0.10,
    subplot_titles=[
        "X acceleration", "X FFT",
        "Y acceleration", "Y FFT",
        "Z acceleration", "Z FFT"
    ]
)

for i in range(3):
    for j, col in enumerate([1, 1, 2]):
        fig.add_trace(initial_traces[i * 3 + j], row=i + 1, col=col)

    fig.update_yaxes(
        title_text="m/s²",
        row=i + 1, col=1
    )
    fig.update_yaxes(
        title_text="Amplitude [m/s²]",
        row=i + 1, col=2
    )

    fig.update_xaxes(range=[0, WINDOW_S], row=i + 1, col=1)
    fig.update_xaxes(range=[0, FS / 2], row=i + 1, col=2)


# Animation frames
frames = []

for j, start_s in enumerate(starts):
    traces, title = window_traces(start_s)

    frames.append(go.Frame(
        name=str(j),
        data=traces,
        traces=list(range(9)),
        layout=go.Layout(title_text=title)
    ))

fig.frames = frames


# Native Plotly slider
steps = []

for j, start_s in enumerate(starts):
    steps.append(dict(
        method="animate",
        label=f"{start_s:.1f}",
        args=[
            [str(j)],
            {
                "mode": "immediate",
                "frame": {"duration": 0, "redraw": True},
                "transition": {"duration": 0}
            }
        ]
    ))

fig.update_layout(
    height=850,
    template="plotly_white",
    showlegend=False,
    title=initial_title,
    sliders=[dict(
        active=0,
        steps=steps,
        currentvalue={"prefix": "Start [s]: "},
        pad={"t": 50}
    )]
)

fig.update_xaxes(title_text="Time [s]", row=3, col=1)
fig.update_xaxes(title_text="Frequency [Hz]", row=3, col=2)

# Display only once
fig.show(renderer="vscode")